In [0]:

# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


"""
Transform equipment events, unit-test results, and tester logs into typed
Silver operational datasets. Invalid records are retained in source-specific
quarantine tables with complete rejection reasons and lineage.
"""

CATALOG = "semiconplus_portfolio"

TABLES = {
    "bronze_equipment": f"{CATALOG}.bronze.equipment_events",
    "bronze_tests": f"{CATALOG}.bronze.unit_test_results",
    "bronze_logs": f"{CATALOG}.bronze.tester_logs_raw",
    "silver_equipment": f"{CATALOG}.silver.equipment_events",
    "silver_tests": f"{CATALOG}.silver.unit_test_results",
    "silver_logs": f"{CATALOG}.silver.tester_logs",
    "quarantine_equipment": f"{CATALOG}.quarantine.equipment_events",
    "quarantine_tests": f"{CATALOG}.quarantine.unit_test_results",
    "quarantine_logs": f"{CATALOG}.quarantine.tester_logs",
    "quality": f"{CATALOG}.monitoring.data_quality_results",
    "lots": f"{CATALOG}.silver.production_lots",
    "devices": f"{CATALOG}.silver.devices",
    "sites": f"{CATALOG}.silver.sites",
    "equipment": f"{CATALOG}.silver.equipment",
}

EXPECTED_BRONZE_COUNTS = {
    "bronze_equipment": 255_640,
    "bronze_tests": 181_250,
    "bronze_logs": 18_125,
}

PIPELINE_RUN_ID = str(uuid4())
PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")

In [0]:
# ===================================================
# BLOCK 2 — DEPENDENCY AND CONTROL-TOTAL CHECKS
# ===================================================

"""
Confirm that every required Bronze source and Silver reference dependency
is available and that Bronze input counts match the approved initial load.
"""

required_tables = list(TABLES.values())[:3] + [
    TABLES["lots"],
    TABLES["devices"],
    TABLES["sites"],
    TABLES["equipment"],
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

assert not missing_tables, f"Required tables are missing: {missing_tables}"

for source_key, expected_count in EXPECTED_BRONZE_COUNTS.items():
    actual_count = spark.table(TABLES[source_key]).count()
    print(f"{source_key}: {actual_count:,}")
    assert actual_count == expected_count

print("Operational-data dependencies passed.")

In [0]:
# ===================================================
# BLOCK 3 — PREPARE REFERENCE KEYS
# ===================================================

"""
Prepare the validated Silver keys used to enforce lot, device, site, and
equipment relationships across the operational datasets.
"""

lot_keys_df = spark.table(TABLES["lots"]).select("lot_id").distinct()
device_keys_df = spark.table(TABLES["devices"]).select("device_id").distinct()
site_keys_df = spark.table(TABLES["sites"]).select("site_id").distinct()

equipment_keys_df = (
    spark.table(TABLES["equipment"])
    .select("equipment_id", "site_id")
    .distinct()
)

In [0]:
# ===================================================
# BLOCK 4 — TYPE AND STANDARDIZE EQUIPMENT EVENTS
# ===================================================

"""
Convert equipment-event timestamps and durations to analytical types and
standardize categorical fields used by equipment-performance reporting.
"""

VALID_EVENT_TYPES = [
    "RUN",
    "IDLE",
    "SETUP",
    "ALARM",
    "MAINTENANCE",
    "PLANNED_DOWNTIME",
    "UNPLANNED_DOWNTIME",
]

equipment_typed_df = (
    spark.table(TABLES["bronze_equipment"])
    .select(
        F.upper(F.trim("event_id")).alias("event_id"),
        F.expr("try_cast(trim(event_timestamp_utc) AS timestamp)").alias(
            "event_timestamp_utc"
        ),
        F.upper(F.trim("site_id")).alias("site_id"),
        F.upper(F.trim("equipment_id")).alias("equipment_id"),
        F.upper(F.trim("event_type")).alias("event_type"),
        F.expr("try_cast(trim(duration_seconds) AS bigint)").alias(
            "duration_seconds"
        ),
        F.upper(F.trim("alarm_code")).alias("alarm_code"),
        F.upper(F.trim("source_system")).alias("source_system"),
        "_rescued_data",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
)

In [0]:
# ===================================================
# BLOCK 5 — VALIDATE EQUIPMENT EVENTS
# ===================================================   

"""
Apply required-field, duration, state-domain, and site-equipment hierarchy
controls to each equipment-event record.
"""

equipment_reference_df = equipment_keys_df.withColumn(
    "_equipment_reference_valid", F.lit(True)
)

equipment_checked_df = (
    equipment_typed_df
    .join(
        F.broadcast(equipment_reference_df),
        ["equipment_id", "site_id"],
        "left",
    )
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("event_id").isNull() | (F.col("event_id") == ""),
                F.lit("MISSING_EVENT_ID"),
            ),
            F.when(
                F.col("event_timestamp_utc").isNull(),
                F.lit("INVALID_EVENT_TIMESTAMP"),
            ),
            F.when(
                F.col("site_id").isNull() | (F.col("site_id") == ""),
                F.lit("MISSING_SITE_ID"),
            ),
            F.when(
                F.col("equipment_id").isNull()
                | (F.col("equipment_id") == ""),
                F.lit("MISSING_EQUIPMENT_ID"),
            ),
            F.when(
                F.col("_equipment_reference_valid").isNull(),
                F.lit("INVALID_SITE_EQUIPMENT_RELATIONSHIP"),
            ),
            F.when(
                ~F.col("event_type").isin(VALID_EVENT_TYPES),
                F.lit("INVALID_EVENT_TYPE"),
            ),
            F.when(
                F.col("duration_seconds").isNull()
                | (F.col("duration_seconds") <= 0),
                F.lit("INVALID_DURATION_SECONDS"),
            ),
            F.when(
                F.col("_rescued_data").isNotNull(),
                F.lit("UNEXPECTED_SOURCE_FIELDS"),
            ),
        )),
    )
    .drop("_equipment_reference_valid")
)


In [0]:
# ===================================================
# BLOCK 6 — PARSE UNIT-TEST JSON FIELDS
# ===================================================   

"""
Parse the JSON strings embedded in each Bronze unit-test record into typed
columns at the declared grain of one test result per tested unit.
"""

lot_schema = StructType([
    StructField("lot_id", StringType(), True),
    StructField("unit_sequence", IntegerType(), True),
])

product_schema = StructType([
    StructField("device_id", StringType(), True),
    StructField("product_group_id", StringType(), True),
])

result_schema = StructType([
    StructField("status", StringType(), True),
    StructField("defect_code", StringType(), True),
    StructField("test_time_seconds", DoubleType(), True),
])

context_schema = StructType([
    StructField("site_id", StringType(), True),
    StructField("equipment_id", StringType(), True),
    StructField("program_revision", StringType(), True),
])

unit_tests_parsed_df = (
    spark.table(TABLES["bronze_tests"])
    .withColumn("_lot", F.from_json("lot", lot_schema))
    .withColumn("_product", F.from_json("product", product_schema))
    .withColumn("_result", F.from_json("result", result_schema))
    .withColumn("_context", F.from_json("test_context", context_schema))
    .select(
        F.upper(F.trim("test_result_id")).alias("test_result_id"),
        F.expr("try_cast(trim(event_timestamp_utc) AS timestamp)").alias(
            "event_timestamp_utc"
        ),
        F.upper(F.trim("_lot.lot_id")).alias("lot_id"),
        F.col("_lot.unit_sequence").cast("int").alias("unit_sequence"),
        F.upper(F.trim("_product.device_id")).alias("device_id"),
        F.upper(F.trim("_product.product_group_id")).alias(
            "product_group_id"
        ),
        F.upper(F.trim("_result.status")).alias("test_status"),
        F.upper(F.trim("_result.defect_code")).alias("defect_code"),
        F.col("_result.test_time_seconds").cast("double").alias(
            "test_time_seconds"
        ),
        F.upper(F.trim("_context.site_id")).alias("site_id"),
        F.upper(F.trim("_context.equipment_id")).alias("equipment_id"),
        F.upper(F.trim("_context.program_revision")).alias(
            "program_revision"
        ),
        "lot",
        "product",
        "result",
        "test_context",
        "_rescued_data",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
)

In [0]:
# ===================================================
# BLOCK 7 — VALIDATE UNIT-TEST RESULTS
# ===================================================   

"""
Apply required-field, domain, duration, and reference-integrity controls
to the flattened unit-test dataset.
"""

unit_test_reference_df = (
    unit_tests_parsed_df.alias("t")
    .join(
        F.broadcast(lot_keys_df.withColumn("_lot_valid", F.lit(True))),
        "lot_id",
        "left",
    )
    .join(
        F.broadcast(
            device_keys_df.withColumn("_device_valid", F.lit(True))
        ),
        "device_id",
        "left",
    )
    .join(
        F.broadcast(
            equipment_keys_df.withColumn(
                "_equipment_valid", F.lit(True)
            )
        ),
        ["equipment_id", "site_id"],
        "left",
    )
)

unit_tests_checked_df = (
    unit_test_reference_df
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("test_result_id").isNull()
                | (F.col("test_result_id") == ""),
                F.lit("MISSING_TEST_RESULT_ID"),
            ),
            F.when(
                F.col("event_timestamp_utc").isNull(),
                F.lit("INVALID_EVENT_TIMESTAMP"),
            ),
            F.when(
                F.col("lot_id").isNull() | (F.col("lot_id") == ""),
                F.lit("MISSING_LOT_ID"),
            ),
            F.when(F.col("_lot_valid").isNull(), F.lit("UNKNOWN_LOT_ID")),
            F.when(
                F.col("unit_sequence").isNull()
                | (F.col("unit_sequence") <= 0),
                F.lit("INVALID_UNIT_SEQUENCE"),
            ),
            F.when(
                F.col("_device_valid").isNull(),
                F.lit("UNKNOWN_DEVICE_ID"),
            ),
            F.when(
                F.col("_equipment_valid").isNull(),
                F.lit("INVALID_SITE_EQUIPMENT_RELATIONSHIP"),
            ),
            F.when(
                ~F.col("test_status").isin("PASS", "FAIL"),
                F.lit("INVALID_TEST_STATUS"),
            ),
            F.when(
                F.col("test_time_seconds").isNull()
                | (F.col("test_time_seconds") <= 0),
                F.lit("INVALID_TEST_TIME_SECONDS"),
            ),
            F.when(
                F.col("program_revision").isNull()
                | (F.col("program_revision") == ""),
                F.lit("MISSING_PROGRAM_REVISION"),
            ),
            F.when(
                F.col("_rescued_data").isNotNull(),
                F.lit("UNEXPECTED_SOURCE_FIELDS"),
            ),
        )),
    )
    .drop("_lot_valid", "_device_valid", "_equipment_valid")
)

In [0]:
# ===================================================
# BLOCK 8 — PARSE TESTER LOGS
# ===================================================   

"""
Parse tester log lines into typed operational fields using the approved
key-value log contract. The raw line remains available for investigation.
"""

LOG_PATTERN = (
    r"^(\S+)\s+(\S+)\s+lot=(\S+)\s+device=(\S+)\s+"
    r"equipment=(\S+)\s+started=(\d+)\s+passed=(\d+)\s+failed=(\d+)\s*$"
)

tester_logs_parsed_df = (
    spark.table(TABLES["bronze_logs"])
    .select(
        F.expr(
            f"try_cast(regexp_extract(value, '{LOG_PATTERN}', 1) AS timestamp)"
        ).alias("event_timestamp_utc"),
        F.upper(F.regexp_extract("value", LOG_PATTERN, 2)).alias("log_level"),
        F.upper(F.regexp_extract("value", LOG_PATTERN, 3)).alias("lot_id"),
        F.upper(F.regexp_extract("value", LOG_PATTERN, 4)).alias("device_id"),
        F.upper(F.regexp_extract("value", LOG_PATTERN, 5)).alias(
            "equipment_id"
        ),
        F.expr(
            f"try_cast(regexp_extract(value, '{LOG_PATTERN}', 6) AS bigint)"
        ).alias("quantity_started"),
        F.expr(
            f"try_cast(regexp_extract(value, '{LOG_PATTERN}', 7) AS bigint)"
        ).alias("quantity_passed"),
        F.expr(
            f"try_cast(regexp_extract(value, '{LOG_PATTERN}', 8) AS bigint)"
        ).alias("quantity_failed"),
        F.regexp_extract("equipment_id", r"^(SITE\d+)-", 1).alias("site_id"),
        F.col("value").alias("raw_log_line"),
        "_rescued_data",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
        F.col("_pipeline_run_id").alias("_bronze_pipeline_run_id"),
    )
)

In [0]:
# ===================================================
# BLOCK 9 — VALIDATE TESTER LOGS
# =================================================== 


"""
Apply parsing, required-key, quantity-reconciliation, and reference checks
to each tester-log record.
"""

tester_logs_reference_df = (
    tester_logs_parsed_df
    .join(
        F.broadcast(lot_keys_df.withColumn("_lot_valid", F.lit(True))),
        "lot_id",
        "left",
    )
    .join(
        F.broadcast(
            device_keys_df.withColumn("_device_valid", F.lit(True))
        ),
        "device_id",
        "left",
    )
    .join(
        F.broadcast(
            equipment_keys_df.withColumn(
                "_equipment_valid", F.lit(True)
            )
        ),
        ["equipment_id", "site_id"],
        "left",
    )
)

tester_logs_checked_df = (
    tester_logs_reference_df
    .withColumn(
        "_quality_reasons",
        F.array_compact(F.array(
            F.when(
                F.col("event_timestamp_utc").isNull(),
                F.lit("LOG_PATTERN_OR_TIMESTAMP_INVALID"),
            ),
            F.when(F.col("log_level") != "INFO", F.lit("INVALID_LOG_LEVEL")),
            F.when(F.col("_lot_valid").isNull(), F.lit("UNKNOWN_LOT_ID")),
            F.when(
                F.col("_device_valid").isNull(),
                F.lit("UNKNOWN_DEVICE_ID"),
            ),
            F.when(
                F.col("_equipment_valid").isNull(),
                F.lit("INVALID_SITE_EQUIPMENT_RELATIONSHIP"),
            ),
            F.when(
                F.col("quantity_started").isNull()
                | (F.col("quantity_started") <= 0),
                F.lit("INVALID_QUANTITY_STARTED"),
            ),
            F.when(
                F.col("quantity_passed") + F.col("quantity_failed")
                != F.col("quantity_started"),
                F.lit("QUANTITY_RECONCILIATION_FAILED"),
            ),
        )),
    )
    .drop("_lot_valid", "_device_valid", "_equipment_valid")
)

In [0]:
# ===================================================
# BLOCK 10 — SPLIT, RECONCILE, AND WRITE OUTPUTS
# =================================================== 

"""
Split every checked source into accepted and quarantined populations,
reconcile all Bronze input records, and replace the current Silver outputs.
"""

datasets = {
    "equipment_events": {
        "checked": equipment_checked_df,
        "silver": TABLES["silver_equipment"],
        "quarantine": TABLES["quarantine_equipment"],
        "expected_source": 255_640,
    },
    "unit_test_results": {
        "checked": unit_tests_checked_df,
        "silver": TABLES["silver_tests"],
        "quarantine": TABLES["quarantine_tests"],
        "expected_source": 181_250,
    },
    "tester_logs": {
        "checked": tester_logs_checked_df,
        "silver": TABLES["silver_logs"],
        "quarantine": TABLES["quarantine_logs"],
        "expected_source": 18_125,
    },
}

quality_metrics = []

for dataset_name, config in datasets.items():
    checked_df = config["checked"]

    accepted_df = (
        checked_df
        .filter(F.size("_quality_reasons") == 0)
        .drop("_quality_reasons")
        .withColumn("_silver_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("_silver_processed_at_utc", F.current_timestamp())
    )

    quarantined_df = (
        checked_df
        .filter(F.size("_quality_reasons") > 0)
        .withColumn("_silver_pipeline_run_id", F.lit(PIPELINE_RUN_ID))
        .withColumn("_quarantined_at_utc", F.current_timestamp())
    )

    source_count = checked_df.count()
    accepted_count = accepted_df.count()
    rejected_count = quarantined_df.count()

    assert source_count == config["expected_source"]
    assert accepted_count + rejected_count == source_count

    (
        accepted_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(config["silver"])
    )

    (
        quarantined_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(config["quarantine"])
    )

    assert spark.table(config["silver"]).count() == accepted_count
    assert spark.table(config["quarantine"]).count() == rejected_count

    quality_metrics.append(
        (dataset_name, source_count, accepted_count, rejected_count)
    )

display(
    spark.createDataFrame(
        quality_metrics,
        ["dataset", "source_rows", "accepted_rows", "rejected_rows"],
    ).orderBy("dataset")
)

In [0]:
# ===================================================
# BLOCK 11 — REVIEW QUARANTINE REASONS
# =================================================== 

"""
Summarize record-level rejection reasons before declaring the Silver
operational layer valid.
"""

reason_frames = []

for dataset_name, config in datasets.items():
    reason_frames.append(
        spark.table(config["quarantine"])
        .select(
            F.lit(dataset_name).alias("dataset_name"),
            F.explode("_quality_reasons").alias("quality_reason"),
        )
    )

combined_reasons_df = reason_frames[0]
for reason_df in reason_frames[1:]:
    combined_reasons_df = combined_reasons_df.unionByName(reason_df)

reason_summary_df = (
    combined_reasons_df
    .groupBy("dataset_name", "quality_reason")
    .count()
    .orderBy("dataset_name", "quality_reason")
)

display(reason_summary_df)

# The generated dataset contains one deliberate invalid equipment duration.
assert spark.table(TABLES["quarantine_equipment"]).filter(
    F.array_contains("_quality_reasons", "INVALID_DURATION_SECONDS")
).count() == 1

print("Expected invalid equipment-duration record detected.")

In [0]:
# ===================================================
# BLOCK 12 — WRITE DATA-QUALITY METRICS
# =================================================== 
"""
Append the current operational-data quality results to the monitoring
history for later workflow dashboards and audit review.
"""

completed_at_utc = datetime.now(timezone.utc)

quality_results_df = spark.createDataFrame(
    [
        (
            PIPELINE_RUN_ID,
            dataset_name,
            source_count,
            accepted_count,
            rejected_count,
            "PASSED",
            completed_at_utc,
        )
        for dataset_name, source_count, accepted_count, rejected_count
        in quality_metrics
    ],
    """
    pipeline_run_id STRING,
    dataset_name STRING,
    source_row_count LONG,
    accepted_row_count LONG,
    rejected_row_count LONG,
    validation_status STRING,
    validated_at_utc TIMESTAMP
    """,
)

(
    quality_results_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLES["quality"])
)

display(quality_results_df.orderBy("dataset_name"))

In [0]:
# ===================================================
# BLOCK 13 — FINAL PIPELINE RESULT
# =================================================== 

"""
Publish the completed Silver operational-data result for execution logs
and validation evidence.
"""

PIPELINE_END_TIME = datetime.now(timezone.utc)

print("SILVER OPERATIONAL-DATA PIPELINE PASSED")
for row in quality_metrics:
    print(
        f"{row[0]}: source={row[1]:,}, "
        f"accepted={row[2]:,}, rejected={row[3]:,}"
    )
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")
print(
    "Duration seconds: "
    f"{(PIPELINE_END_TIME - PIPELINE_START_TIME).total_seconds():.2f}"
)